In [0]:
%pip install geopandas pyogrio shapely

In [0]:
# Databricks notebook source
# =============================================================
#  AgriMap · GeoSafras — Gerador de Rotas (Databricks)
#
#  Lê o SHP de talhões do Volume e grava:
#    1) gs_gold.rotas_tableau  → tabela Delta (lugar do CSV)
#    2) gs_gold.rotas_resumo   → tabela Delta (lugar do rotas_resumo.csv)
#    3) rotas_maquina.shp      → Volume (para embarcação nas máquinas)
#
#  SHP de entrada:
#    /Volumes/workspace/pipeline_estudo/raw_files/geo_safras/
#    {FAZENDA}/shp/{ARQUIVO_TALHOES}
# =============================================================

# COMMAND ----------
# %pip install geopandas pyogrio shapely
# Descomente a linha acima na PRIMEIRA execução para instalar as bibliotecas

# COMMAND ----------
import os
import glob
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
from shapely.affinity import rotate
from shapely.ops import unary_union
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime

# =============================================================
#  CONFIG — ajuste aqui para cada fazenda
# =============================================================
FAZENDA_ID      = "geo_safras"
BASE_PATH       = "/Volumes/workspace/pipeline_estudo/raw_files/geo_safras"
SHP_PATH        = "/Volumes/workspace/pipeline_estudo/raw_files/geo_safras/shp"
OUTPUT_SHP_PATH = "/Volumes/workspace/pipeline_estudo/raw_files/geo_safras/shp/rotas_maquina.shp"

ARQUIVO_TALHOES    = "geo_safras.shp"
ARQUIVO_OBSTACULOS = None

CAMPO_NOME    = "subcultura"
CAMPO_FAZENDA = "localidade"
CAMPO_CULTURA = "cultura"

COPIAR_TODOS_CAMPOS = True
CAMPOS_EXTRAS = []

# Parâmetros operacionais
LARGURA_FAIXA   = 9.0
VELOCIDADE_KMH  = 7.0
TEMPO_MANOBRA_S = 30.0
RECUO_BORDADURA = 8.0

# Produtividade e logística
PRODUTIVIDADE_TON_HA_1   = 4.0
PRODUTIVIDADE_TON_HA_2   = 6.0
PRODUTIVIDADE_CULTURA = {
    "soja":  (3.2, 3.9),
    "milho": (6.0, 9.0),
    "sorgo": (2.8, 3.8),
}
CAPACIDADE_GRANELEIRO_TON    = 12.0
TEMPO_DESCARGA_PARADA_MIN    = 2.0
TEMPO_DESCARGA_MOVIMENTO_MIN = 0.5
SACA_KG          = 60.0
CAP_CAMINHAO_TON = 36.0
JORNADA_DIARIA_H = 10.0

ESTRATEGIA_MAQUINA   = "melhor"
ESTRATEGIAS_BASE     = ["horizontal", "vertical", "otima"]
INCLUIR_CIRCULAR     = True
INCLUIR_CONTORNO     = False

FATOR_VISUAL_TABLEAU = 16
FATOR_VISUAL_MAQUINA = 1

# Databricks
CATALOG  = "workspace"
SCHEMA_G = "gs_gold"
NOW      = datetime.now().isoformat()

# =============================================================
#  NÚCLEO — idêntico ao gerar_rotas.py local (não alterar)
# =============================================================

# COMMAND ----------
def area_de_trabalho(geom, obstaculos=None, recuo=0.0):
    poly = geom
    if recuo and recuo > 0:
        poly = poly.buffer(-recuo)
    if obstaculos:
        poly = poly.difference(unary_union(obstaculos))
    return poly

def gerar_faixas(area, largura, ang):
    c = area.centroid
    g = rotate(area, -ang, origin=c)
    minx, miny, maxx, maxy = g.bounds
    faixas, y, passada = [], miny + largura / 2.0, 0
    while y <= maxy:
        corte = LineString([(minx - 1, y), (maxx + 1, y)]).intersection(g)
        if not corte.is_empty:
            segs = [corte] if corte.geom_type == "LineString" else \
                   list(corte.geoms) if corte.geom_type == "MultiLineString" else []
            for s in segs:
                if s.length > 0:
                    faixas.append((passada, rotate(s, ang, origin=c)))
        y += largura
        passada += 1
    return faixas

def _so_geoms(faixas):
    return [g for _, g in faixas]

def metricas_rota(area, largura, ang, vel, tman):
    faixas = gerar_faixas(area, largura, ang)
    geoms = _so_geoms(faixas)
    comp = sum(s.length for s in geoms)
    n = len(geoms)
    tp = (comp / 1000.0) / vel
    tm = max(n - 1, 0) * tman / 3600.0
    tt = tp + tm
    return {"angulo": round(ang, 1), "passadas": n, "manobras": max(n - 1, 0),
            "tempo_produtivo_h": round(tp, 3), "tempo_manobra_h": round(tm, 3),
            "tempo_total_h": round(tt, 3), "dist_m": round(comp, 1),
            "eficiencia_campo": round(tp / tt, 4) if tt > 0 else 0.0,
            "faixas": faixas}

def rumo_otimo(area, largura, vel, tman, passo=2):
    melhor = None
    for a in range(0, 180, passo):
        m = metricas_rota(area, largura, a, vel, tman)
        if melhor is None or m["tempo_total_h"] < melhor["tempo_total_h"]:
            melhor = m
    return melhor

def metricas_circular(area, largura, vel, t_trans=8.0):
    c = area.centroid
    raio = max(Point(p).distance(c) for p in area.exterior.coords)
    aneis, comp, r, anel = [], 0.0, largura / 2.0, 0
    while r <= raio:
        corte = c.buffer(r, resolution=64).exterior.intersection(area)
        if not corte.is_empty:
            comp += corte.length
            aneis.append((anel, c.buffer(r, resolution=64).exterior))
        r += largura
        anel += 1
    n = len(aneis)
    tp = (comp / 1000.0) / vel
    tm = max(n - 1, 0) * t_trans / 3600.0
    tt = tp + tm
    return {"aneis": n, "transicoes": max(n - 1, 0),
            "tempo_produtivo_h": round(tp, 3), "tempo_manobra_h": round(tm, 3),
            "tempo_total_h": round(tt, 3), "dist_m": round(comp, 1),
            "eficiencia_campo": round(tp / tt, 4) if tt > 0 else 0.0,
            "geom_aneis": aneis}

def metricas_contorno(area, largura, vel, t_trans=8.0):
    aneis, comp, d, idx = [], 0.0, largura / 2.0, 0
    while idx < 100000:
        inner = area.buffer(-d)
        if inner.is_empty or inner.area <= 0:
            break
        parts = [inner] if inner.geom_type == "Polygon" else \
                list(inner.geoms) if inner.geom_type == "MultiPolygon" else []
        if not parts:
            break
        achou = False
        for poly in parts:
            for ring in [poly.exterior] + list(poly.interiors):
                if ring.length > 0:
                    aneis.append((idx, LineString(ring.coords)))
                    comp += ring.length
                    achou = True
        if not achou:
            break
        d += largura
        idx += 1
    n = len(aneis)
    tp = (comp / 1000.0) / vel
    tm = max(n - 1, 0) * t_trans / 3600.0
    tt = tp + tm
    return {"aneis": n, "transicoes": max(n - 1, 0),
            "tempo_produtivo_h": round(tp, 3), "tempo_manobra_h": round(tm, 3),
            "tempo_total_h": round(tt, 3), "dist_m": round(comp, 1),
            "eficiencia_campo": round(tp / tt, 4) if tt > 0 else 0.0,
            "geom_aneis": aneis}

def eh_circular(geom, limiar=0.90):
    p = geom.length
    return p > 0 and (4 * math.pi * geom.area) / (p * p) >= limiar

def estrategias_candidatas(geom):
    estr = list(ESTRATEGIAS_BASE)
    if INCLUIR_CIRCULAR and eh_circular(geom):
        estr.append("circular")
    elif INCLUIR_CONTORNO:
        estr.append("contorno")
    return estr

def _obst_do_talhao(geom, obst_gdf):
    if obst_gdf is None or obst_gdf.empty:
        return None
    sel = obst_gdf[obst_gdf.intersects(geom)]
    return list(sel.geometry) if not sel.empty else None

def _espiral(area, espacamento):
    c = area.centroid
    raio = max(Point(p).distance(c) for p in area.exterior.coords)
    b = espacamento / (2 * math.pi)
    pts, theta = [], 0.0
    while True:
        r = b * theta
        if r > raio:
            break
        pts.append((c.x + r * math.cos(theta), c.y + r * math.sin(theta)))
        theta += max(0.03, min(0.4, 4.0 / (r + 1.0)))
    if len(pts) < 2:
        return []
    corte = LineString(pts).intersection(area)
    if corte.is_empty:
        return []
    segs = [corte] if corte.geom_type == "LineString" else \
           list(corte.geoms) if corte.geom_type == "MultiLineString" else []
    return [(i, s) for i, s in enumerate(segs) if s.length > 0]

def _linhas_metricas(geom, estr, obst, fator=1):
    area = area_de_trabalho(geom, obst, RECUO_BORDADURA)
    if estr == "otima":
        m = rumo_otimo(area, LARGURA_FAIXA, VELOCIDADE_KMH, TEMPO_MANOBRA_S)
        return m["faixas"], m
    if estr == "circular":
        m = metricas_circular(area, LARGURA_FAIXA, VELOCIDADE_KMH)
        return _espiral(area, LARGURA_FAIXA * max(fator, 1)), m
    if estr == "contorno":
        m = metricas_contorno(area, LARGURA_FAIXA, VELOCIDADE_KMH)
        return m["geom_aneis"], m
    ang = 0 if estr == "horizontal" else 90
    m = metricas_rota(area, LARGURA_FAIXA, ang, VELOCIDADE_KMH, TEMPO_MANOBRA_S)
    return m["faixas"], m

def _bous(linhas):
    pts, rev = [], False
    for fid, ln in enumerate(linhas):
        cs = list(ln.coords)[::-1] if rev else list(ln.coords)
        for ordem, (x, y) in enumerate(cs):
            pts.append((ordem, fid, x, y))
        rev = not rev
    return pts

def _decimar(faixas, fator):
    if not fator or fator <= 1:
        return faixas
    return [(pi, g) for (pi, g) in faixas if pi % fator == 0]

def _norm_cultura(texto):
    import unicodedata
    s = unicodedata.normalize("NFKD", str(texto)).encode("ascii","ignore").decode().lower()
    for chave in PRODUTIVIDADE_CULTURA:
        if chave in s:
            return chave
    return None

def _produtividade(row, nome):
    fonte = ""
    if CAMPO_CULTURA and CAMPO_CULTURA in row and pd.notna(row[CAMPO_CULTURA]):
        fonte = row[CAMPO_CULTURA]
    chave = _norm_cultura(fonte) or _norm_cultura(nome)
    return PRODUTIVIDADE_CULTURA.get(chave, (PRODUTIVIDADE_TON_HA_1, PRODUTIVIDADE_TON_HA_2))

def _logistica(area_ha, tp_h, tman_h, prod1=PRODUTIVIDADE_TON_HA_1, prod2=PRODUTIVIDADE_TON_HA_2):
    base = tp_h + tman_h
    out = {}
    for tag, prod_ha in (("p1", prod1), ("p2", prod2)):
        producao = area_ha * prod_ha
        desc = producao / CAPACIDADE_GRANELEIRO_TON if CAPACIDADE_GRANELEIRO_TON > 0 else 0.0
        t_par = desc * TEMPO_DESCARGA_PARADA_MIN / 60.0
        t_mov = desc * TEMPO_DESCARGA_MOVIMENTO_MIN / 60.0
        tt_par, tt_mov = base + t_par, base + t_mov
        out[f"producao_ton_{tag}"]   = round(producao, 1)
        out[f"descargas_{tag}"]      = round(desc, 1)
        out[f"t_total_parada_{tag}"] = round(tt_par, 2)
        out[f"t_total_movim_{tag}"]  = round(tt_mov, 2)
        out[f"rend_op_parada_{tag}"] = round(tp_h / tt_par * 100, 1) if tt_par > 0 else 0.0
        out[f"rend_op_movim_{tag}"]  = round(tp_h / tt_mov * 100, 1) if tt_mov > 0 else 0.0
        out[f"producao_kg_{tag}"]    = round(producao * 1000.0, 0)
        out[f"sacas_{tag}"]          = round(producao * 1000.0 / SACA_KG, 0) if SACA_KG > 0 else 0.0
        out[f"graneleiros_{tag}"]    = int(math.ceil(producao / CAPACIDADE_GRANELEIRO_TON)) if CAPACIDADE_GRANELEIRO_TON > 0 else 0
        out[f"caminhoes_{tag}"]      = int(math.ceil(producao / CAP_CAMINHAO_TON)) if CAP_CAMINHAO_TON > 0 else 0
    return out

def _cobertura(dist_m, area_ha):
    if not dist_m or area_ha <= 0:
        return {"cobertura_pct": 0.0, "sobreposicao_pct": 0.0}
    area_coberta = (dist_m * LARGURA_FAIXA) / 10000.0
    cob = area_coberta / area_ha * 100.0
    sob = max(0.0, (area_coberta - area_ha) / area_ha * 100.0)
    return {"cobertura_pct": round(cob, 1), "sobreposicao_pct": round(sob, 1)}

def _lower(v):
    return v.lower().strip() if isinstance(v, str) else v

def gerar_tabelas(gdf, obst_gdf):
    pontos, resumo = [], []
    _reservados = {"faixa","ordem","x","y","estrategia","rend_pct","voltas",
                   "passadas","manobras","tempo_produtivo_h","tempo_manobra_h",
                   "tempo_total_h","dist_m","lon","lat","id", gdf.geometry.name}
    cols_copiar = [c for c in gdf.columns if c not in _reservados] if COPIAR_TODOS_CAMPOS else \
                  [c for c in CAMPOS_EXTRAS if c in gdf.columns]
    for _, row in gdf.iterrows():
        nome  = _lower(row[CAMPO_NOME])
        faz   = _lower(row[CAMPO_FAZENDA]) if CAMPO_FAZENDA else ""
        extras = {c: _lower(row[c]) for c in cols_copiar}
        geom   = row.geometry
        area_ha_t = float(row["area_ha"]) if ("area_ha" in row and pd.notna(row["area_ha"])) else geom.area / 10000.0
        obst   = _obst_do_talhao(geom, obst_gdf)
        for estr in estrategias_candidatas(geom):
            linhas, m = _linhas_metricas(geom, estr, obst, FATOR_VISUAL_TABLEAU)
            rend    = round(m["eficiencia_campo"] * 100, 1)
            voltas  = m.get("passadas", m.get("aneis"))
            manobras= m.get("manobras", m.get("transicoes"))
            tp      = m.get("tempo_produtivo_h")
            tman    = m.get("tempo_manobra_h")
            ind = {"rend_pct": rend, "voltas": voltas, "manobras": manobras,
                   "angulo_graus": m.get("angulo"),
                   "tempo_produtivo_h": tp, "tempo_manobra_h": tman,
                   "tempo_total_h": m["tempo_total_h"], "dist_m": m.get("dist_m"),
                   "dias_colheita": round(m["tempo_total_h"] / JORNADA_DIARIA_H, 2) if JORNADA_DIARIA_H > 0 else None,
                   **_cobertura(m.get("dist_m"), area_ha_t),
                   **_logistica(area_ha_t, tp or 0.0, tman or 0.0, *_produtividade(row, nome))}
            linhas_vis = _decimar(linhas, FATOR_VISUAL_TABLEAU)
            for (ordem, faixa, x, y) in _bous(_so_geoms(linhas_vis)):
                pontos.append({**extras, "fazenda": faz, "talhao": nome,
                               "estrategia": estr, "faixa": faixa, "ordem": ordem,
                               "x": round(x, 3), "y": round(y, 3), **ind})
            resumo.append({**extras, "fazenda": faz, "talhao": nome,
                           "estrategia": estr, "angulo_graus": m.get("angulo"),
                           "passadas": voltas, "manobras": manobras,
                           "tempo_total_h": m["tempo_total_h"], "rend_pct": rend})
    df_pts, df_res = pd.DataFrame(pontos), pd.DataFrame(resumo)
    if not df_pts.empty:
        sc_cols = [c for c in ("sacas_p1","sacas_p2") if c in df_pts.columns]
        if sc_cols:
            base = df_pts.drop_duplicates(subset=["fazenda","talhao"])[["fazenda"] + sc_cols]
            tot  = (base.groupby("fazenda")[sc_cols].sum()
                        .rename(columns={c: c.replace("sacas_","sacas_total_") for c in sc_cols})
                        .reset_index())
            df_pts = df_pts.merge(tot, on="fazenda", how="left")
            if not df_res.empty:
                df_res = df_res.merge(tot, on="fazenda", how="left")
    for df in (df_pts, df_res):
        if not df.empty:
            df["id"] = (df["fazenda"].astype(str) + " · " + df["talhao"].astype(str)
                        if CAMPO_FAZENDA else df["talhao"].astype(str))
    if not df_pts.empty:
        pg = gpd.GeoDataFrame(df_pts.copy(),
                              geometry=gpd.points_from_xy(df_pts["x"], df_pts["y"]),
                              crs=gdf.crs).to_crs(epsg=4326)
        df_pts["lon"] = pg.geometry.x.round(7)
        df_pts["lat"] = pg.geometry.y.round(7)
    return df_pts, df_res

def _escolha_para(idv, df_res):
    if isinstance(ESTRATEGIA_MAQUINA, dict):
        return ESTRATEGIA_MAQUINA.get(idv, "otima")
    if ESTRATEGIA_MAQUINA == "melhor":
        sub = df_res[df_res["id"] == idv]
        return sub.loc[sub["rend_pct"].idxmax(), "estrategia"] if not sub.empty else "otima"
    return ESTRATEGIA_MAQUINA

def gerar_gdf_rotas(gdf, obst_gdf, df_res):
    geoms, attrs = [], []
    for _, row in gdf.iterrows():
        nome = _lower(row[CAMPO_NOME])
        faz  = _lower(row[CAMPO_FAZENDA]) if CAMPO_FAZENDA else ""
        idv  = (f"{faz} · {nome}" if CAMPO_FAZENDA else str(nome))
        estr = _escolha_para(idv, df_res)
        geom = row.geometry
        obst = _obst_do_talhao(geom, obst_gdf)
        linhas, m = _linhas_metricas(geom, estr, obst, FATOR_VISUAL_MAQUINA)
        rend     = round(m["eficiencia_campo"] * 100, 1)
        voltas   = int(m.get("passadas", m.get("aneis")))
        manobras = int(m.get("manobras", m.get("transicoes")))
        tempo_h  = float(m["tempo_total_h"])
        linhas_vis = _decimar(linhas, FATOR_VISUAL_MAQUINA)
        ordem, rev = 0, False
        for fid, ln in enumerate(_so_geoms(linhas_vis)):
            cs = list(ln.coords)[::-1] if rev else list(ln.coords)
            geoms.append(LineString(cs))
            attrs.append({"fazenda": str(faz)[:50], "talhao": str(nome)[:50],
                          "estr": estr[:10], "faixa": int(fid), "ordem": int(ordem),
                          "largura": float(LARGURA_FAIXA), "rend_pct": float(rend),
                          "angulo": float(m["angulo"]) if m.get("angulo") is not None else -1.0,
                          "voltas": voltas, "manobras": manobras, "tempo_h": tempo_h})
            ordem += 1; rev = not rev
    return gpd.GeoDataFrame(attrs, geometry=geoms, crs=gdf.crs)

# COMMAND ----------
# =============================================================
#  MAIN — adaptado para Databricks
# =============================================================
def main():
    print(f"[Rotas] Início: {NOW}")
    print(f"[Rotas] SHP path: {SHP_PATH}")

    # 1. Localizar SHP
    if ARQUIVO_TALHOES:
        caminho_talhoes = f"{SHP_PATH}/{ARQUIVO_TALHOES}"
    else:
        shps = glob.glob(f"{SHP_PATH}/*.shp")
        caminho_talhoes = shps[0] if shps else None

    if not caminho_talhoes or not os.path.exists(caminho_talhoes):
        raise SystemExit(f"[ERRO] SHP não encontrado em: {SHP_PATH}")
    print(f"[OK] Talhões: {os.path.basename(caminho_talhoes)}")

    # 2. Ler SHP
    gdf = gpd.read_file(caminho_talhoes, engine="pyogrio")
    crs_orig = gdf.crs
    print(f"[OK] {len(gdf)} talhões | CRS: {crs_orig.to_epsg() if crs_orig else 'indefinido'}")

    # 3. Reprojetar para CRS métrico se necessário
    if crs_orig is None:
        raise SystemExit("[ERRO] SHP sem CRS (.prj ausente).")
    if crs_orig.is_geographic:
        crs_metrico = gdf.estimate_utm_crs()
        gdf_calc = gdf.to_crs(crs_metrico)
        print(f"[INFO] Reprojetado para cálculo: EPSG:{crs_metrico.to_epsg()}")
    else:
        gdf_calc = gdf
        print("[INFO] CRS já é métrico.")

    # 4. Obstáculos (opcional)
    obst_gdf = None
    if ARQUIVO_OBSTACULOS:
        cam = f"{SHP_PATH}/{ARQUIVO_OBSTACULOS}"
        if os.path.exists(cam):
            obst_gdf = gpd.read_file(cam, engine="pyogrio").to_crs(gdf_calc.crs)
            print(f"[OK] Obstáculos: {len(obst_gdf)}")

    # 5. Gerar tabelas
    df_pts, df_res = gerar_tabelas(gdf_calc, obst_gdf)
    print(f"[OK] Pontos gerados: {len(df_pts)} | Resumo: {len(df_res)}")

    # 6. Adicionar metadados de auditoria
    df_pts["fazenda_id"]         = FAZENDA_ID
    df_pts["ingestion_timestamp"] = NOW
    df_res["fazenda_id"]         = FAZENDA_ID
    df_res["ingestion_timestamp"] = NOW

    # 7. Salvar rotas_tableau como tabela Delta (MERGE por fazenda_id+talhao+estrategia+faixa+ordem)
    spark_pts = spark.createDataFrame(df_pts)
    if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_G}.rotas_tableau"):
        dt = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA_G}.rotas_tableau")
        chave = """
            tgt.fazenda_id = src.fazenda_id AND
            tgt.talhao     = src.talhao     AND
            tgt.estrategia = src.estrategia AND
            tgt.faixa      = src.faixa      AND
            tgt.ordem      = src.ordem
        """
        (dt.alias("tgt")
           .merge(spark_pts.alias("src"), chave)
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())
        print("  ✓ rotas_tableau: MERGE concluído")
    else:
        spark_pts.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_G}.rotas_tableau")
        print("  ✓ rotas_tableau: criada e carregada")

    # 8. Salvar rotas_resumo como tabela Delta (MERGE por fazenda_id+talhao+estrategia)
    spark_res = spark.createDataFrame(df_res)
    if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_G}.rotas_resumo"):
        dt2 = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA_G}.rotas_resumo")
        chave2 = """
            tgt.fazenda_id = src.fazenda_id AND
            tgt.talhao     = src.talhao     AND
            tgt.estrategia = src.estrategia
        """
        (dt2.alias("tgt")
            .merge(spark_res.alias("src"), chave2)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print("  ✓ rotas_resumo: MERGE concluído")
    else:
        spark_res.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_G}.rotas_resumo")
        print("  ✓ rotas_resumo: criada e carregada")

    # 9. Salvar rotas_maquina.shp no Volume
    gdf_rotas = gerar_gdf_rotas(gdf_calc, obst_gdf, df_res)
    if crs_orig.is_geographic:
        gdf_rotas = gdf_rotas.to_crs(crs_orig)

    # Remover SHP antigo se existir
    base = os.path.splitext(OUTPUT_SHP_PATH)[0]
    for ext in (".shp", ".shx", ".dbf", ".prj", ".cpg"):
        f = base + ext
        if os.path.exists(f):
            os.remove(f)

    gdf_rotas.to_file(OUTPUT_SHP_PATH, driver="ESRI Shapefile", engine="pyogrio")
    print(f"  ✓ rotas_maquina.shp: {len(gdf_rotas)} linhas | CRS: {gdf_rotas.crs.to_epsg()}")
    print(f"  ✓ Disponível em: {OUTPUT_SHP_PATH}")

    print(f"\n[Rotas] ✅ Concluído em {datetime.now().isoformat()}")
    print(f"  Tabelas disponíveis:")
    print(f"  • {CATALOG}.{SCHEMA_G}.rotas_tableau")
    print(f"  • {CATALOG}.{SCHEMA_G}.rotas_resumo")
    print(f"  • SHP: {OUTPUT_SHP_PATH}")

# COMMAND ----------
main()